# Gemma 4 E4B Legal Merge + VLM Validation (A100)

**Goal**: Merge the trained legal LoRA adapter into the regular Gemma 4 E4B multimodal base on an A100,
while preserving the original vision, audio, and multimodal projector weights.

**This notebook is the regular Gemma 4 path**:
- Base model: `google/gemma-4-E4B-it`
- Canonical output: merged Hugging Face safetensors checkpoint
- Validation target: multimodal VLM prompts after merge
- TensorRT-LLM and Triton should consume the merged HF checkpoint, not GGUF
- LiteRT conversion and eval are intentionally handled in a separate notebook/workflow

## Why this path on A100?

Google's Gemma Hugging Face docs show the standard Transformers flow for Gemma 4,
and the Unsloth Gemma 4 guidance indicates E4B can run in full 16-bit precision on high-memory GPUs.
On an A100, it is simpler and safer to merge directly into the regular BF16 base than to
load a 4-bit checkpoint and dequantize it just to save merged weights again.

## Architecture
```
google/gemma-4-E4B-it (regular multimodal base, BF16 on A100)
  ├── language_model              ← apply legal LoRA adapter here
  ├── vision_tower               ← KEEP ORIGINAL (frozen, never trained)
  ├── audio_tower                ← KEEP ORIGINAL (frozen, never trained)
  └── multi_modal_projector      ← KEEP ORIGINAL
```

## Key Rule

There is **no manual tensor re-attach step** if the full regular Gemma 4 base is loaded.
The correct workflow is:
1. Strip the adapter down to language-only LoRA tensors
2. Load the full regular Gemma 4 base
3. Apply only the language adapter
4. Merge and save the full multimodal model
5. Validate image understanding with the merged model
6. Branch from the merged HF checkpoint into TRT-LLM, GGUF, or LiteRT workflows

## Prerequisites
- **Runtime**: Colab A100 80GB recommended for the cleanest BF16 merge path
- **Adapter**: `Semaj90/gemma4-e4b-legal-grpo` on HF Hub or a local uploaded stripped adapter
- **HF Token**: Set in Colab Secrets as `HF_TOKEN`

## Steps
1. Install core dependencies for Gemma 4 + PEFT merge
2. Log into Hugging Face
3. Download the adapter and strip it to language-only LoRA
4. Load the regular Gemma 4 E4B base in BF16 on A100
5. Apply the legal adapter and confirm no multimodal LoRA remains
6. Merge and save the full multimodal checkpoint
7. Run a VLM smoke test against the merged model
8. Stage a TRT-LLM/Triton export bundle from merged safetensors
9. Export GGUF later if needed for llama.cpp or Ollama
10. Handle LiteRT eval in a separate follow-up notebook

## 1. Install Dependencies

In [ ]:
# Core Colab dependencies only.
# Keep llama.cpp out of the first cell so GPU/RAM stay focused on Unsloth + merge work.
import sys
import subprocess

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'unsloth', 'mergekit', 'mergekit-moe', '-y'], check=False)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '--upgrade', '--no-cache-dir',
        'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
    ],
    check=True
 )
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install',
        'bitsandbytes', 'accelerate', 'peft', 'transformers',
        'huggingface_hub', 'pillow', 'safetensors', 'requests'
    ],
    check=True
 )

import torch

# Pin torchao to avoid torch.int1 AttributeError (requires PyTorch 2.7+)
torch_major, torch_minor = [int(x) for x in torch.__version__.split('.')[:2]]
if torch_major < 2 or (torch_major == 2 and torch_minor < 7):
    print(f'PyTorch {torch.__version__} detected (<2.7) — pinning torchao==0.7.0')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torchao==0.7.0', '--quiet'], check=False)
else:
    print(f'PyTorch {torch.__version__} — torchao version OK')

print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"})')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB' if torch.cuda.is_available() else '')
print('\nllama.cpp setup is deferred until the GGUF export cell.')
print('If imports fail after package installation in a fresh Colab session, restart the runtime once and rerun from Cell 1.')

## 2. HuggingFace Login

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('Logged in via Colab Secrets')
except Exception:
    print('Set HF_TOKEN in Colab Secrets (key icon in sidebar)')
    login()

## 3. Download Adapter + Strip to Language-Only LoRA

This notebook assumes the adapter should affect **language reasoning only**.
The full regular Gemma 4 base will supply the original vision, audio, and projector weights unchanged.

**Two paths** — pick ONE:

### Path A: Upload pre-stripped adapter from local machine
If you already have `gemma4-legal-text-only-adapter/adapter_model.safetensors`, upload it directly and set `USE_LOCAL_ADAPTER = True`.

### Path B: Download the GRPO adapter from HF and strip it automatically
Downloads `Semaj90/gemma4-e4b-legal-grpo`, removes any LoRA tensors targeting:
- `vision_tower`
- `audio_tower`
- `multi_modal_projector`

The result is a language-only adapter that can be safely merged into the regular Gemma 4 multimodal base on A100.

In [ ]:
import os, json, shutil
from safetensors.torch import load_file, save_file

# ============================================================
# TOGGLE: Set True if you uploaded the pre-stripped adapter
#         from c:/Users/james/Downloads/gemma4-legal-text-only-adapter/
# ============================================================
USE_LOCAL_ADAPTER = False

ADAPTER_DIR = 'gemma4-e4b-legal-grpo-lora'
TEXT_ONLY_DIR = 'gemma4-e4b-legal-text-only-adapter'
BLOCKED_MODULE_FRAGMENTS = ('vision_tower', 'audio_tower', 'multi_modal_projector')

def enforce_language_only_config(config_path):
    with open(config_path) as f:
        config = json.load(f)

    original_targets = config.get('target_modules')
    removed_targets = []
    if isinstance(original_targets, list):
        filtered_targets = [
            name for name in original_targets
            if not any(fragment in name for fragment in BLOCKED_MODULE_FRAGMENTS)
        ]
        removed_targets = [name for name in original_targets if name not in filtered_targets]
        if not filtered_targets:
            raise ValueError('Filtered target_modules is empty. Fix adapter_config.json before merge.')
        config['target_modules'] = filtered_targets
    else:
        print('target_modules is not a list; leaving it unchanged. Verify adapter_config.json manually before merge.')

    config['exclude_modules'] = list(BLOCKED_MODULE_FRAGMENTS)
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)

    return removed_targets, config.get('target_modules')

if USE_LOCAL_ADAPTER:
    # ---- Path A: Use uploaded pre-stripped adapter ----
    assert os.path.exists(f'{TEXT_ONLY_DIR}/adapter_model.safetensors'), (
        f'Upload adapter_model.safetensors to {TEXT_ONLY_DIR}/ first!\n'
        'Local path: c:/Users/james/Downloads/gemma4-legal-text-only-adapter/'
    )
    assert os.path.exists(f'{TEXT_ONLY_DIR}/adapter_config.json'), (
        f'Upload adapter_config.json to {TEXT_ONLY_DIR}/ alongside the safetensors file!\n'
        'PEFT needs both files to load the adapter correctly.'
    )

    removed_targets, kept_targets = enforce_language_only_config(
        f'{TEXT_ONLY_DIR}/adapter_config.json'
    )
    tensors = load_file(f'{TEXT_ONLY_DIR}/adapter_model.safetensors')
    print('Path A: Using uploaded pre-stripped adapter')
    print(f'  {len(tensors)} tensors ({sum(v.nelement() * v.element_size() for v in tensors.values()) / 1024**2:.1f} MB)')
    vis = [k for k in tensors if 'vision_tower' in k]
    aud = [k for k in tensors if 'audio_tower' in k]
    proj = [k for k in tensors if 'multi_modal_projector' in k]
    print(f'  vision_tower:          {len(vis)} (should be 0)')
    print(f'  audio_tower:           {len(aud)} (should be 0)')
    print(f'  multi_modal_projector: {len(proj)} (should be 0)')
    if removed_targets:
        print(f'  target_modules trimmed: removed {len(removed_targets)} multimodal entries')
    print(f'  target_modules kept:   {len(kept_targets) if isinstance(kept_targets, list) else kept_targets}')
    del tensors

else:
    # ---- Path B: Download from HF Hub + strip ----
    if not os.path.exists(f'{ADAPTER_DIR}/adapter_model.safetensors'):
        print('Downloading adapter from HF Hub...')
        from huggingface_hub import snapshot_download
        snapshot_download('Semaj90/gemma4-e4b-legal-grpo', local_dir=ADAPTER_DIR)
        print(f'Downloaded to {ADAPTER_DIR}/')
    else:
        print(f'Adapter already at {ADAPTER_DIR}/')

    tensors = load_file(f'{ADAPTER_DIR}/adapter_model.safetensors')
    keys = sorted(tensors.keys())
    lang = {k: v for k, v in tensors.items() if 'language_model' in k}
    vis = [k for k in keys if 'vision_tower' in k]
    aud = [k for k in keys if 'audio_tower' in k]
    proj = [k for k in keys if 'multi_modal_projector' in k]

    print(f'\nOriginal adapter: {len(keys)} tensors')
    print(f'  language_model:        {len(lang)}')
    print(f'  vision_tower:          {len(vis)} (untrained — will strip LoRA only)')
    print(f'  audio_tower:           {len(aud)} (untrained — will strip LoRA only)')
    print(f'  multi_modal_projector: {len(proj)} (untrained — will strip LoRA only)')

    os.makedirs(TEXT_ONLY_DIR, exist_ok=True)
    save_file(lang, f'{TEXT_ONLY_DIR}/adapter_model.safetensors')

    with open(f'{ADAPTER_DIR}/adapter_config.json') as f:
        config = json.load(f)
    with open(f'{TEXT_ONLY_DIR}/adapter_config.json', 'w') as f:
        json.dump(config, f, indent=2)

    removed_targets, kept_targets = enforce_language_only_config(
        f'{TEXT_ONLY_DIR}/adapter_config.json'
    )

    for fname in os.listdir(ADAPTER_DIR):
        if fname not in ('adapter_model.safetensors', 'adapter_config.json'):
            src = os.path.join(ADAPTER_DIR, fname)
            if os.path.isfile(src):
                shutil.copy2(src, os.path.join(TEXT_ONLY_DIR, fname))

    orig_mb = sum(v.nelement() * v.element_size() for v in tensors.values()) / 1024**2
    new_mb = sum(v.nelement() * v.element_size() for v in lang.values()) / 1024**2
    print(f'\nAdapter surgery: {len(keys)} -> {len(lang)} tensors')
    print(f'  {orig_mb:.1f} MB -> {new_mb:.1f} MB (saved {orig_mb - new_mb:.1f} MB)')
    if removed_targets:
        print(f'  target_modules trimmed: removed {len(removed_targets)} multimodal entries')
    print(f'  target_modules kept:   {len(kept_targets) if isinstance(kept_targets, list) else kept_targets}')
    print(f'\nLanguage LoRA saved to: {TEXT_ONLY_DIR}/')
    print('Vision/audio/projector weights will come from the FULL BASE MODEL (original, untouched)')

    del tensors, lang

stripped_tensors = load_file(f'{TEXT_ONLY_DIR}/adapter_model.safetensors')
bad_keys = [
    key for key in stripped_tensors
    if any(fragment in key for fragment in BLOCKED_MODULE_FRAGMENTS)
]
assert not bad_keys, f'Stripped adapter still has multimodal tensors: {bad_keys[:5]}'
del stripped_tensors

print(f'\n=== Ready for merge ===')
print(f'Text-only adapter: {TEXT_ONLY_DIR}/adapter_model.safetensors')
print(f'Adapter config:    {TEXT_ONLY_DIR}/adapter_config.json')
print('Base model will supply the original frozen vision/audio/projector tensors during merge.')

## 4. Load Regular Gemma 4 Base + Apply Legal Adapter

This section switches to the **regular Gemma 4 E4B Hugging Face checkpoint** on A100.
That keeps the merge target in standard HF format from the start.

We do **not** dequantize a 4-bit Unsloth base here.
Instead, we load `google/gemma-4-E4B-it` in BF16, attach the language-only legal adapter,
and verify that no LoRA layers touch vision/audio/projector modules.

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoProcessor
from peft import PeftModel

BASE_MODEL_ID = 'google/gemma-4-E4B-it'
TEXT_ONLY_DIR = 'gemma4-e4b-legal-text-only-adapter'
DTYPE = torch.bfloat16

assert torch.cuda.is_available(), 'A CUDA GPU is required for this notebook.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {gpu_name}')
print(f'VRAM: {vram_gb:.1f} GB')
if 'A100' in gpu_name:
    print('GPU OK: A100 is the primary tested path for Unsloth + BF16 merge.')
elif any(x in gpu_name for x in ('PRO 6000', 'G4', 'Blackwell', 'B200', 'GB200')):
    print('NOTE: Blackwell GPU detected. A100 is the recommended runtime for this notebook.')
    print('      Blackwell (sm_120) is NOT yet fully supported by Unsloth/bitsandbytes as of April 2026.')
    print('      Expect possible CUDA or triton kernel errors. Switch to A100 Pro+ if they occur.')
    print('      G4 Blackwell is better for LiteRT inference (§9) than for the Unsloth merge.')
else:
    print(f'WARNING: {gpu_name} is not a known-good GPU for this notebook.')
    print('         A100 80GB (Colab Pro+) is the recommended runtime.')

assert os.path.exists(f'{TEXT_ONLY_DIR}/adapter_model.safetensors'), \
    f'Text-only adapter not found at {TEXT_ONLY_DIR}/. Run the adapter prep cell first!'
assert os.path.exists(f'{TEXT_ONLY_DIR}/adapter_config.json'), \
    f'adapter_config.json missing from {TEXT_ONLY_DIR}/.'

print(f'Loading regular Gemma 4 base: {BASE_MODEL_ID}')
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=DTYPE,
    device_map='auto',
)
model.eval()
print('Base model loaded in BF16.')

print(f'\nApplying language-only legal adapter from: {TEXT_ONLY_DIR}')
model = PeftModel.from_pretrained(model, TEXT_ONLY_DIR)
model.eval()

lora_params = [n for n, p in model.named_parameters() if 'lora_' in n]
print(f'LoRA parameters loaded: {len(lora_params)}')
print(f'  language_model:        {sum(1 for n in lora_params if "language_model" in n)}')
print(f'  vision_tower:          {sum(1 for n in lora_params if "vision_tower" in n)}')
print(f'  audio_tower:           {sum(1 for n in lora_params if "audio_tower" in n)}')
print(f'  multi_modal_projector: {sum(1 for n in lora_params if "multi_modal_projector" in n)}')

bad_lora = [
    n for n in lora_params
    if any(fragment in n for fragment in ('vision_tower', 'audio_tower', 'multi_modal_projector'))
]

if bad_lora:
    raise RuntimeError(
        f'Adapter still contains multimodal LoRA hooks: {bad_lora[:5]}'
    )

print('\nReady to run multimodal validation and then merge into the regular Gemma 4 checkpoint.')


## 5. Quick Vision Inference Test (Pre-Merge)

Verify the model can process images before committing to the merge.

In [ ]:
from PIL import Image
import requests
from io import BytesIO

# Download a legal-document-style test image
test_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3b/Constitution_of_the_United_States%2C_page_1.jpg/800px-Constitution_of_the_United_States%2C_page_1.jpg'
try:
    img_data = requests.get(test_url, timeout=10).content
    test_image = Image.open(BytesIO(img_data)).convert('RGB')
    print(f'Test image: {test_image.size} ({len(img_data) / 1024:.0f} KB)')
except Exception as e:
    print(f'Image download failed: {e}')
    test_image = Image.new('RGB', (768, 768), (200, 200, 200))
    print('Using placeholder image')

prompt = (
    'Analyze this legal document image. Identify the document type, '
    'summarize visible text or structure, and note any legal-significant details.'
)

messages = [
    {'role': 'user', 'content': [
        {'type': 'image', 'image': test_image},
        {'type': 'text', 'text': prompt},
    ]}
 ]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors='pt',
    enable_thinking=False,
 ).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
        do_sample=True,
    )

response = processor.decode(
    outputs[0][inputs['input_ids'].shape[1]:],
    skip_special_tokens=True
 )
print(f'\n=== VLM Response (adapter attached, pre-merge) ===\n{response}')

## 6. Merge Adapter Into Regular Gemma 4 and Save Merged HF Checkpoint

On A100, the clean merge path is:
1. Load `google/gemma-4-E4B-it` in BF16
2. Apply the stripped legal adapter
3. Run `merge_and_unload()`
4. Save a standard merged Hugging Face checkpoint
5. Re-open that merged checkpoint for VLM validation and downstream export

This notebook is now optimized for the regular Gemma 4 merge path.
LiteRT export and evaluation will be handled separately after this merged checkpoint is validated.

In [ ]:
import os
import gc
import json
import torch

CLEAN_DIR = 'gemma4-legal-vlm-merged'
os.makedirs(CLEAN_DIR, exist_ok=True)

assert torch.cuda.is_available(), 'A CUDA GPU is required for merge.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'Merge GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)')
if 'A100' in gpu_name:
    print('Merge GPU OK: A100 is the primary tested path.')
elif any(x in gpu_name for x in ('PRO 6000', 'G4', 'Blackwell', 'B200', 'GB200')):
    print('NOTE: Blackwell GPU detected for merge step.')
    print('      merge_and_unload() itself (pure PyTorch) should work — no Unsloth kernels involved here.')
    print('      If this errors, re-run §4 load on a fresh A100 runtime first.')
else:
    print(f'WARNING: {gpu_name} — A100 80GB is the recommended merge runtime.')

gc.collect()
torch.cuda.empty_cache()

print('Merging legal adapter into regular Gemma 4 base...')
model = model.merge_and_unload()
model.eval()

total = sum(p.numel() for p in model.parameters())
vision_params = sum(p.numel() for n, p in model.named_parameters() if 'vision_tower' in n)
audio_params = sum(p.numel() for n, p in model.named_parameters() if 'audio_tower' in n)
proj_params = sum(p.numel() for n, p in model.named_parameters() if 'multi_modal_projector' in n)
lang_params = sum(p.numel() for n, p in model.named_parameters() if 'language_model' in n)

print(f'Merged model params: {total:,}')
print(f'  language_model:        {lang_params:,}')
print(f'  vision_tower:          {vision_params:,}')
print(f'  audio_tower:           {audio_params:,}')
print(f'  multi_modal_projector: {proj_params:,}')

if vision_params == 0 or proj_params == 0:
    raise RuntimeError('Merged model is missing required multimodal weights.')

print(f'\nSaving merged HF checkpoint to {CLEAN_DIR}/ ...')
model.save_pretrained(
    CLEAN_DIR,
    safe_serialization=True,
    max_shard_size='5GB',
)
processor.save_pretrained(CLEAN_DIR)

try:
    model.generation_config.save_pretrained(CLEAN_DIR)
except Exception as e:
    print(f'generation_config save skipped: {e}')

manifest = {
    'base_model': BASE_MODEL_ID,
    'adapter_dir': TEXT_ONLY_DIR,
    'format': 'huggingface-safetensors',
    'dtype': 'bfloat16',
    'target_runtime': 'a100-bf16-merge',
    'modalities_preserved': {
        'vision_tower': vision_params > 0,
        'audio_tower': audio_params > 0,
        'multi_modal_projector': proj_params > 0,
    },
}
with open(os.path.join(CLEAN_DIR, 'merge_manifest.json'), 'w') as f:
    json.dump(manifest, f, indent=2)

saved_files = []
for root, _, files in os.walk(CLEAN_DIR):
    for name in files:
        path = os.path.join(root, name)
        saved_files.append((path, os.path.getsize(path)))

total_gb = sum(size for _, size in saved_files) / 1024**3
print(f'\nSaved {len(saved_files)} files ({total_gb:.1f} GB total) to {CLEAN_DIR}/')
print('Merged regular Gemma 4 checkpoint is ready for clean reload and VLM validation.')


## 7. Reload Merged Checkpoint and Validate VLM

Reload the saved merged checkpoint from disk and run the same document-image prompt again.
This proves the persisted regular Gemma 4 checkpoint still supports multimodal inference after merge.

If this step works on A100, the merged safetensors checkpoint becomes the canonical source artifact for:
- TensorRT-LLM and Triton text-serving workflows
- optional GGUF export for llama.cpp or Ollama
- future LiteRT packaging and eval work

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

CLEAN_DIR = 'gemma4-legal-vlm-merged'
assert os.path.exists(CLEAN_DIR), f'Merged checkpoint missing at {CLEAN_DIR}/'

print(f'Reloading merged checkpoint from {CLEAN_DIR}/ ...')
processor = AutoProcessor.from_pretrained(CLEAN_DIR)
model = AutoModelForCausalLM.from_pretrained(
    CLEAN_DIR,
    torch_dtype=torch.bfloat16,
    device_map='auto',
 )
model.eval()

reload_messages = [
    {'role': 'user', 'content': [
        {'type': 'image', 'image': test_image},
        {'type': 'text', 'text': (
            'Analyze this legal document image. Identify the document type, '
            'summarize visible content, and explain why the text appears legally relevant.'
        )},
    ]}
 ]

reload_inputs = processor.apply_chat_template(
    reload_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors='pt',
    enable_thinking=False,
 ).to(model.device)

with torch.no_grad():
    reload_outputs = model.generate(
        **reload_inputs,
        max_new_tokens=256,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
        do_sample=True,
    )

reload_response = processor.decode(
    reload_outputs[0][reload_inputs['input_ids'].shape[1]:],
    skip_special_tokens=True
 )
print(f'\n=== VLM Response (merged checkpoint reload) ===\n{reload_response}')

## 8. Prepare TRT-LLM / Triton Export Bundle

This section keeps the merged Hugging Face checkpoint as the source of truth and writes a small staging bundle
for your local TensorRT-LLM and Triton workflow.

Use this branch for fast **text inference**, long-context serving, paged KV cache, and direct comparison against
your local TRT endpoint at `http://localhost:8099`.

Notes:
- TRT-LLM consumes the merged HF checkpoint, not GGUF
- your app currently talks to TRT-LLM through `/health` and `/v1/completions` on port `8099`
- VLM behavior should still be validated from the merged HF model or GGUF runtime unless and until you prove
  a matching multimodal TRT path end-to-end

In [ ]:
import os
import json
import textwrap

CLEAN_DIR = 'gemma4-legal-vlm-merged'
TRT_EXPORT_DIR = 'gemma4-legal-vlm-trt-export'
os.makedirs(TRT_EXPORT_DIR, exist_ok=True)

assert os.path.exists(CLEAN_DIR), f'Merged checkpoint missing at {CLEAN_DIR}/'

readme = textwrap.dedent(f'''
    # Gemma 4 Legal TRT-LLM / Triton Bundle

    Source checkpoint: {CLEAN_DIR}/
    Runtime target: TensorRT-LLM text serving behind Triton or the standalone TRT endpoint on port 8099.


    ## 1. Convert merged HF checkpoint to TRT-LLM checkpoint

    python examples/gemma/convert_checkpoint.py \\
      --model_dir /path/to/{CLEAN_DIR} \\
      --output_dir /path/to/trt_checkpoints/gemma4_legal \\
      --dtype bfloat16 \\
      --tp_size 1 \\
      --pp_size 1

    ## 2. Build engine with paged KV cache

    trtllm-build \\
      --checkpoint_dir /path/to/trt_checkpoints/gemma4_legal \\
      --output_dir /path/to/trt_engines/gemma4_legal \\
      --gemm_plugin auto \\
      --gpt_attention_plugin auto \\
      --max_batch_size 4 \\
      --max_input_len 4096 \\
      --max_output_len 1024 \\
      --paged_kv_cache enable \\
      --context_fmha enable \\
      --remove_input_padding enable

    ## 3. Smoke test your existing local endpoint

    curl http://localhost:8099/health

    curl -X POST http://localhost:8099/v1/completions \\
      -H 'Content-Type: application/json' \\
      -d '{{
        "prompt": "Summarize the evidentiary value of a notarized affidavit in two short paragraphs.",
        "max_tokens": 256,
        "temperature": 0.2,
        "stream": false
      }}'

    ## 4. Compare tool-call-friendly prompting

    Prompt file: tool_call_prompt.txt
    Goal: compare TRT text output against LiteRT tool-call output and merged-HF VLM output.
''').strip() + '\n'

smoke_prompt = (
    'Summarize the evidentiary value of a notarized affidavit in two short paragraphs. '
    'Focus on authentication, hearsay limits, and chain-of-custody implications.'
)

tool_call_prompt = (
    'You may call tools if needed. Available tools: glossary_search(query), case_search(query). '
    'Return either a direct answer or a structured tool call for the best next legal research step on '
    'chain of custody defects in a criminal case.'
)

with open(os.path.join(TRT_EXPORT_DIR, 'README.md'), 'w', encoding='utf-8') as f:
    f.write(readme)

with open(os.path.join(TRT_EXPORT_DIR, 'smoke_prompt.txt'), 'w', encoding='utf-8') as f:
    f.write(smoke_prompt + '\n')

with open(os.path.join(TRT_EXPORT_DIR, 'tool_call_prompt.txt'), 'w', encoding='utf-8') as f:
    f.write(tool_call_prompt + '\n')

bundle_manifest = {
    'source_checkpoint': CLEAN_DIR,
    'source_format': 'huggingface-safetensors',
    'target_runtime': 'trt-llm-triton-text',
    'local_endpoint': 'http://localhost:8099',
    'health_path': '/health',
    'completion_path': '/v1/completions',
    'notes': [
        'Use merged HF checkpoint as the canonical source artifact.',
        'Use GGUF only for llama.cpp or Ollama branches.',
        'Validate VLM separately from TRT text-serving until multimodal TRT wiring is proven.'
    ]
}

with open(os.path.join(TRT_EXPORT_DIR, 'bundle_manifest.json'), 'w', encoding='utf-8') as f:
    json.dump(bundle_manifest, f, indent=2)

print(f'TRT export bundle written to {TRT_EXPORT_DIR}/')
print('Files: README.md, smoke_prompt.txt, tool_call_prompt.txt, bundle_manifest.json')
print('Use the merged HF checkpoint for TRT-LLM conversion, then compare against the local 8099 endpoint.')

## 9. Convert Merged Checkpoint → LiteRT (.litertlm)

Converts the merged HF safetensors checkpoint into a `.litertlm` artifact for LiteRT-LM inference.

**This is the missing link between the A100 merge and the LiteRT eval notebook.**

### Pipeline after this cell
```
gemma4-legal-vlm-merged/  (safetensors, this notebook)
  ↓ ai-edge-torch export
gemma4-legal-vlm-litert/gemma4-legal.litertlm
  ↓ upload to HF Hub  (Semaj90/gemma4-legal-litert-lm)
  ↓
LiteRT eval notebook (T4 Colab, free tier)
  CUSTOM_HF_REPO = 'Semaj90/gemma4-legal-litert-lm'
  CUSTOM_HF_FILE = 'gemma4-legal.litertlm'
```

### Requirements
- A100 with 40+ GB VRAM (conversion loads the full BF16 model)
- `ai-edge-torch[generative]` — Google's PyTorch → LiteRT exporter
- This step is **text-only** — the `.litertlm` format targets phones/edge devices and does not carry the full vision tower.
  Keep the merged HF checkpoint as the canonical multimodal source.

> **Note:** The HF Gemma 4 → `.litertlm` conversion path was stabilised in `ai-edge-torch` >= 0.3.
> If this cell fails on a given Colab runtime, check `ai-edge-torch` release notes for Gemma 4 support.


In [ ]:
import gc
import json
import os
import subprocess
import sys
import torch

CLEAN_DIR = 'gemma4-legal-vlm-merged'
LITERT_DIR = 'gemma4-legal-vlm-litert'
LITERT_FILE = 'gemma4-legal.litertlm'
LITERT_PATH = os.path.join(LITERT_DIR, LITERT_FILE)
HF_LITERT_REPO = 'Semaj90/gemma4-legal-litert-lm'

os.makedirs(LITERT_DIR, exist_ok=True)
assert os.path.exists(CLEAN_DIR), (
    f'Merged checkpoint not found at {CLEAN_DIR}/. Run merge cell (§6) first.'
)

# ── Install ────────────────────────────────────────────────────────────────────
print('Installing ai-edge-torch[generative] ...')
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', 'ai-edge-torch[generative]', '-q', '--upgrade'],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT
)
import importlib
import ai_edge_torch

version_str = getattr(ai_edge_torch, '__version__', 'unknown')
print(f'ai-edge-torch version: {version_str}')
try:
    from packaging.version import Version
    if Version(version_str) < Version('0.3.0'):
        print(f'WARNING: {version_str} may not support Gemma 4. Recommend >= 0.3.0.')
except Exception:
    pass

# ── Locate Gemma module (path varies by version) ───────────────────────────────
# 0.3.x: generative.examples.gemma.gemma3 (covers Gemma 3 + 4 family)
# future: may split to gemma4 submodule
gemma_mod = None
for mod_path in (
    'ai_edge_torch.generative.examples.gemma.gemma4',
    'ai_edge_torch.generative.examples.gemma.gemma3',
):
    try:
        gemma_mod = importlib.import_module(mod_path)
        print(f'Gemma module: {mod_path}')
        break
    except ImportError:
        continue

if gemma_mod is None:
    print('\n=== Gemma 4 E4B LiteRT conversion not yet available ===')
    print('ai-edge-torch does not include a Gemma 4 module in this version.')
    print('Options:')
    print('  A. Pre-built: litert-community/gemma-4-E2B-it-litert-lm (use in eval notebook)')
    print('  B. https://github.com/google-ai-edge/ai-edge-torch/releases')
    print('  C. Try again after: pip install ai-edge-torch[generative] --upgrade')
    print('Skipping. §10 GGUF export works independently.')
    sys.exit(0)

# ── Locate E4B config ──────────────────────────────────────────────────────────
model_config = None
for fn_name in ('get_model_config_4b', 'get_model_config_e4b', 'get_model_config_2b'):
    fn = getattr(gemma_mod, fn_name, None)
    if fn is not None:
        model_config = fn()
        print(f'Config: {fn_name}()')
        break

if model_config is None:
    print('ERROR: No E4B config found in Gemma module.')
    print('Available:', [a for a in dir(gemma_mod) if 'config' in a.lower()])
    sys.exit(0)

model_config.max_seq_len = 4096

# ── Build edge model ───────────────────────────────────────────────────────────
print(f'\nBuilding edge model from {CLEAN_DIR}/ ...')
build_fn = getattr(gemma_mod, 'build_model', None)
if build_fn is None:
    print('ERROR: build_model() not found in Gemma module.')
    sys.exit(0)
edge_model = build_fn(model_config, CLEAN_DIR)
edge_model.eval()
print('Edge model ready.')

# ── Export (two API paths — version dependent) ─────────────────────────────────
print(f'\nExporting to {LITERT_PATH} ...')
EXPORT_OK = False

try:
    # Path 1: utilities.export.export_to_litert (ai-edge-torch 0.3+)
    from ai_edge_torch.generative.utilities import export as litert_export
    export_fn = getattr(litert_export, 'export_to_litert', None)
    if export_fn is None:
        raise ImportError('export_to_litert not in utilities.export')
    export_fn(
        pytorch_model=edge_model,
        model_config=model_config,
        output_path=LITERT_PATH,
        prefill_seq_len=256,
        kv_cache_max_len=model_config.max_seq_len,
    )
    EXPORT_OK = True
    print('Path 1 (utilities.export_to_litert) succeeded.')
except Exception as e1:
    print(f'Path 1 failed: {e1}')
    try:
        # Path 2: ai_edge_torch.convert() — older/alternate API
        from ai_edge_torch import convert
        sample_kw = {
            'tokens': torch.zeros((1, 256), dtype=torch.int32),
            'input_pos': torch.arange(256, dtype=torch.int32).unsqueeze(0),
        }
        tflite_model = convert(edge_model.cpu(), (sample_kw,))
        tflite_model.export(LITERT_PATH)
        EXPORT_OK = True
        print('Path 2 (ai_edge_torch.convert) succeeded.')
    except Exception as e2:
        print(f'Path 2 failed: {e2}')

if not EXPORT_OK:
    print('\n=== LiteRT export unavailable for Gemma 4 E4B on this version ===')
    print('Possible causes:')
    print('  - Gemma 4 E4B recipe not yet merged in ai-edge-torch')
    print('  - Static-shape tracing incompatible with multimodal checkpoint')
    print('Options:')
    print('  A. Pre-built: litert-community/gemma-4-E2B-it-litert-lm (eval notebook)')
    print('  B. https://github.com/google-ai-edge/ai-edge-torch/releases')
    print('  C. Wait for ai-edge-torch >= 0.4 with native E4B recipe')
    print('\nProceeding to §10 GGUF is safe — it works independently.')
else:
    size_gb = os.path.getsize(LITERT_PATH) / 1024**3
    print(f'Export: {LITERT_PATH} ({size_gb:.2f} GB)')

    manifest = {
        'source_checkpoint': CLEAN_DIR,
        'litert_file': LITERT_FILE,
        'hf_target_repo': HF_LITERT_REPO,
        'ai_edge_torch_version': version_str,
        'model_config': {'params': '4b', 'max_seq_len': model_config.max_seq_len},
        'note': 'Text-only. Vision/audio towers not in .litertlm format.',
    }
    with open(os.path.join(LITERT_DIR, 'litert_manifest.json'), 'w') as mf:
        json.dump(manifest, mf, indent=2)
    print('Manifest written.')

    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(HF_LITERT_REPO, repo_type='model', exist_ok=True)
    api.upload_file(
        path_or_fileobj=LITERT_PATH,
        path_in_repo=LITERT_FILE,
        repo_id=HF_LITERT_REPO,
        repo_type='model',
    )
    api.upload_file(
        path_or_fileobj=os.path.join(LITERT_DIR, 'litert_manifest.json'),
        path_in_repo='litert_manifest.json',
        repo_id=HF_LITERT_REPO,
        repo_type='model',
    )
    print(f'Uploaded to https://huggingface.co/{HF_LITERT_REPO}')
    print(f'\nIn the LiteRT eval notebook set:')
    print(f'  CUSTOM_HF_REPO = "{HF_LITERT_REPO}"')
    print(f'  CUSTOM_HF_FILE = "{LITERT_FILE}"')

gc.collect()
torch.cuda.empty_cache()


## 10. Prepare llama.cpp + Export Multimodal GGUF (Optional)

Only run this after the merged HF checkpoint has already passed the VLM reload test above.
This section is optional and separate from the main A100 merge validation path.

Gemma 4 multimodal GGUF may require two files:
1. Language model GGUF
2. mmproj GGUF

Use this branch for llama.cpp, Ollama, and TurboQuant-style local serving.


In [ ]:
import os, subprocess, sys

# Prepare llama.cpp only when GGUF export is needed
if not os.path.exists('llama.cpp'):
    print('Cloning llama.cpp for GGUF export...')
    subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git'],
        check=True
    )
else:
    print('Refreshing llama.cpp checkout...')
    subprocess.run(['git', '-C', 'llama.cpp', 'pull'], check=False)

req_path = 'llama.cpp/requirements.txt'
if os.path.exists(req_path):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', req_path], check=False)

CLEAN_DIR = 'gemma4-legal-vlm-merged'
GGUF_DIR = 'gemma4-legal-vlm-gguf'
os.makedirs(GGUF_DIR, exist_ok=True)

bf16_path = os.path.join(GGUF_DIR, 'gemma4-legal-vlm-BF16.gguf')
q4_path = os.path.join(GGUF_DIR, 'gemma4-legal-vlm-Q4_K_M.gguf')
mmproj_path = os.path.join(GGUF_DIR, 'gemma4-legal-vlm-mmproj-BF16.gguf')

# Step 1: Convert safetensors -> BF16 GGUF (full model)
print('Step 1/3: Converting merged BF16 safetensors -> BF16 GGUF...')
result = subprocess.run(
    ['python', 'llama.cpp/convert_hf_to_gguf.py', CLEAN_DIR,
     '--outfile', bf16_path, '--outtype', 'bf16'],
    capture_output=True, text=True
 )
if result.returncode == 0:
    fsize = os.path.getsize(bf16_path)
    print(f'  BF16 GGUF: {fsize / 1024**3:.1f} GB')
else:
    print(f'  ERROR: {result.stderr[:500]}')
    print('  Trying alternative conversion...')
    result = subprocess.run(
        ['python', 'llama.cpp/convert_hf_to_gguf.py', CLEAN_DIR,
         '--outfile', bf16_path, '--outtype', 'bf16', '--model-type', 'gemma4'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        fsize = os.path.getsize(bf16_path)
        print(f'  BF16 GGUF (alt): {fsize / 1024**3:.1f} GB')
    else:
        print(f'  Fallback also failed: {result.stderr[:500]}')

# Step 2: Quantize BF16 -> Q4_K_M
if os.path.exists(bf16_path):
    print('\nStep 2/3: Quantizing BF16 -> Q4_K_M...')
    quantize_bin = 'llama.cpp/build/bin/llama-quantize'
    if not os.path.exists(quantize_bin):
        print('  Building llama-quantize...')
        subprocess.run(
            ['cmake', '-B', 'llama.cpp/build', '-S', 'llama.cpp', '-DCMAKE_BUILD_TYPE=Release'],
            capture_output=True
        )
        subprocess.run(
            ['cmake', '--build', 'llama.cpp/build', '--target', 'llama-quantize', '-j4'],
            capture_output=True
        )

    result = subprocess.run(
        [quantize_bin, bf16_path, q4_path, 'Q4_K_M'],
        capture_output=True, text=True
    )
    if result.returncode == 0 and os.path.exists(q4_path):
        fsize = os.path.getsize(q4_path)
        print(f'  Q4_K_M GGUF: {fsize / 1024**3:.1f} GB')
    else:
        print(f'  Quantization failed: {result.stderr[:500]}')

# Step 3: Extract multimodal projector -> mmproj GGUF
print('\nStep 3/3: Extracting multimodal projector...')
mmproj_script = 'llama.cpp/examples/llava/convert_image_encoder_to_gguf.py'
if not os.path.exists(mmproj_script):
    mmproj_script = 'llama.cpp/tools/mtmd/convert_image_encoder_to_gguf.py'

if os.path.exists(mmproj_script):
    result = subprocess.run(
        ['python', mmproj_script, '--model_dir', CLEAN_DIR,
         '--output_path', mmproj_path],
        capture_output=True, text=True
    )
    if result.returncode == 0 and os.path.exists(mmproj_path):
        fsize = os.path.getsize(mmproj_path)
        print(f'  mmproj GGUF: {fsize / 1024**2:.0f} MB')
    else:
        print(f'  mmproj extraction failed: {result.stderr[:500]}')
        print('  NOTE: mmproj may already be included in the main GGUF for Gemma4')
        print('  Check llama.cpp/docs/multimodal.md for the current Gemma4 path')
else:
    print('  mmproj script not found at expected paths')
    print('  You may need to extract it manually or use a newer llama.cpp checkout')

print('\n=== GGUF Export Summary ===')
for f in [bf16_path, q4_path, mmproj_path]:
    if os.path.exists(f):
        print(f'  {os.path.basename(f)}: {os.path.getsize(f) / 1024**3:.2f} GB')
    else:
        print(f'  {os.path.basename(f)}: NOT CREATED')

## 10. Create Ollama Modelfile

Deploy as `gemma4-legal-vlm:latest` on Ollama with multimodal projector.

In [ ]:
import os

GGUF_DIR = 'gemma4-legal-vlm-gguf'
q4_path = os.path.join(GGUF_DIR, 'gemma4-legal-vlm-Q4_K_M.gguf')
mmproj_path = os.path.join(GGUF_DIR, 'gemma4-legal-vlm-mmproj-BF16.gguf')

# Determine which GGUF to use
model_gguf = q4_path if os.path.exists(q4_path) else os.path.join(GGUF_DIR, 'gemma4-legal-vlm-BF16.gguf')

modelfile = f'''FROM {os.path.basename(model_gguf)}
'''

# Add mmproj if it exists as separate file
if os.path.exists(mmproj_path):
    modelfile += f'''PROJECTOR {os.path.basename(mmproj_path)}
'''

modelfile += '''PARAMETER temperature 0.3
PARAMETER num_predict 4096
PARAMETER num_ctx 32768
PARAMETER stop <end_of_turn>
PARAMETER stop <eos>

TEMPLATE """{{- range .Messages }}
{{- if eq .Role "system" }}<start_of_turn>system
{{ .Content }}<end_of_turn>
{{- else if eq .Role "user" }}<start_of_turn>user
{{ .Content }}<end_of_turn>
{{- else if eq .Role "assistant" }}<start_of_turn>model
{{ .Content }}<end_of_turn>
{{- end }}
{{- end }}<start_of_turn>model
"""

SYSTEM """You are a legal AI assistant specialized in evidence analysis, case law research,
and legal document interpretation. You have been fine-tuned on legal reasoning tasks
including statutory interpretation, case analysis, evidence evaluation, and legal writing.

When analyzing images of legal documents, evidence photos, or exhibits:
- Identify the document type and key elements
- Extract relevant text, dates, signatures, and markings
- Note any anomalies, redactions, or chain-of-custody indicators
- Provide structured analysis suitable for legal proceedings
"""
'''

modelfile_path = os.path.join(GGUF_DIR, 'Modelfile')
with open(modelfile_path, 'w') as f:
    f.write(modelfile)

print('=== Modelfile Created ===')
print(modelfile)
print(f'\nSaved to: {modelfile_path}')
print(f'\n=== Deployment Commands ===')
print(f'cd {GGUF_DIR}')
print(f'ollama create gemma4-legal-vlm:latest -f Modelfile')
print(f'ollama run gemma4-legal-vlm:latest')

## 11. Video Frame Analysis Test

Gemma 4 processes video as a sequence of image frames.
- Configurable visual token budget: 70-1120 tokens per image
- Use low budget (70-140) for video = more frames, faster inference
- Max ~60 seconds of video at 1 FPS native

This cell demonstrates the frame extraction -> batch analysis pattern
used by [mattsvlm](https://github.com/vast-data/mattsvlm).

In [ ]:
# Video frame analysis demo
# In production: extract frames with ffmpeg, send batch to Ollama

import subprocess, os, time
from PIL import Image

def extract_frames(video_path, fps=1, max_frames=30):
    """Extract frames from video at given FPS using ffmpeg."""
    frames_dir = 'video_frames'
    os.makedirs(frames_dir, exist_ok=True)

    cmd = [
        'ffmpeg', '-i', video_path,
        '-vf', f'fps={fps}',
        '-frames:v', str(max_frames),
        '-q:v', '2',
        os.path.join(frames_dir, 'frame_%04d.jpg'),
        '-y'
    ]
    subprocess.run(cmd, capture_output=True)

    frames = sorted([
        os.path.join(frames_dir, f)
        for f in os.listdir(frames_dir)
        if f.endswith('.jpg')
    ])
    return [Image.open(f).convert('RGB') for f in frames[:max_frames]]


def analyze_video_frames(frames, model, processor, prompt=None):
    """Analyze video frames as a batch of images."""
    if prompt is None:
        prompt = (
            'These are sequential frames from a video recording. '
            'Describe what is happening in this footage. '
            'Note any persons, actions, objects, or events relevant to a legal investigation.'
        )

    content = [{'type': 'image', 'image': frame} for frame in frames]
    content.append({'type': 'text', 'text': prompt})

    messages = [{'role': 'user', 'content': content}]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
    ).to(model.device)

    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3,
            do_sample=True,
        )
    elapsed = time.time() - t0

    response = processor.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    return response, elapsed


print('=== Video Frame Analysis Demo ===')
print('Creating synthetic test frames...')

test_frames = []
colors = [(200, 100, 100), (100, 200, 100), (100, 100, 200), (200, 200, 100), (200, 100, 200)]
for color in colors:
    img = Image.new('RGB', (384, 384), color)
    test_frames.append(img)

print(f'Frames: {len(test_frames)} ({test_frames[0].size})')
print('\nNote: In production, use extract_frames() with real video files.')
print('Example: frames = extract_frames("evidence_recording.mp4", fps=1, max_frames=30)')
print('         response, elapsed = analyze_video_frames(frames, model, processor)')
print('\nFor Ollama deployment, send frames as base64 images array:')
print('  curl http://localhost:11434/api/generate -d \'{"model":"gemma4-legal-vlm","images":["<b64>","<b64>",...]}\'')
print('\nRecommended settings for video:')
print('  - Token budget: 70-140 per frame (speed over detail)')
print('  - FPS: 1-2 for surveillance, 0.5 for documents/static')
print('  - Max frames: 30 (Gemma4 handles up to ~60 at low budget)')

## 12. Package for Download

Collects merged HF, TRT staging files, and optional GGUF artifacts for download or Google Drive persistence.

In [ ]:
import os
import shutil

GGUF_DIR = 'gemma4-legal-vlm-gguf'
CLEAN_DIR = 'gemma4-legal-vlm-merged'
TRT_EXPORT_DIR = 'gemma4-legal-vlm-trt-export'
ARTIFACT_DIRS = [CLEAN_DIR, TRT_EXPORT_DIR, GGUF_DIR]

print('=== Output Files ===')
for artifact_dir in ARTIFACT_DIRS:
    if not os.path.exists(artifact_dir):
        print(f'  {artifact_dir}/: NOT PRESENT')
        continue

    for root, _, files in os.walk(artifact_dir):
        rel_root = os.path.relpath(root, '.')
        for name in sorted(files):
            fpath = os.path.join(root, name)
            fsize = os.path.getsize(fpath)
            if fsize > 1024**3:
                size_str = f'{fsize / 1024**3:.2f} GB'
            else:
                size_str = f'{fsize / 1024**2:.1f} MB'
            print(f'  {os.path.join(rel_root, name)}: {size_str}')

print('\n=== Save to Google Drive ===')
try:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_DIR = '/content/drive/MyDrive/gemma4-legal-vlm-artifacts'
    os.makedirs(DRIVE_DIR, exist_ok=True)

    for artifact_dir in ARTIFACT_DIRS:
        if not os.path.exists(artifact_dir):
            continue
        dst_dir = os.path.join(DRIVE_DIR, artifact_dir)
        if os.path.exists(dst_dir):
            shutil.rmtree(dst_dir)
        shutil.copytree(artifact_dir, dst_dir)
        print(f'  Copied {artifact_dir}/ -> {dst_dir}')

    print(f'\nSaved to Google Drive: {DRIVE_DIR}')
except Exception as e:
    print(f'Google Drive not available: {e}')
    print('Use the Colab file browser or zip specific artifact folders manually.')

print('\n=== Local Deployment Branches ===')
print(f'1. Canonical merged HF checkpoint: {CLEAN_DIR}/')
print(f'2. TRT-LLM/Triton staging bundle: {TRT_EXPORT_DIR}/')
if os.path.exists(GGUF_DIR):
    print(f'3. Optional GGUF branch: {GGUF_DIR}/')
    print('   Ollama: ollama create gemma4-legal-vlm:latest -f gemma4-legal-vlm-gguf/Modelfile')
else:
    print('3. Optional GGUF branch: not created yet')

print('\nLiteRT comparison should use the separate LiteRT eval notebook and a .litertlm model artifact.')

## 13. Upload Artifacts to HF Hub (Optional)

Upload the merged HF checkpoint first. Upload GGUF only if you also want a llama.cpp or Ollama distribution.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
CLEAN_DIR = 'gemma4-legal-vlm-merged'
GGUF_DIR = 'gemma4-legal-vlm-gguf'
MERGED_REPO_ID = 'Semaj90/gemma4-e4b-legal-vlm-merged'
GGUF_REPO_ID = 'Semaj90/gemma4-e4b-legal-vlm-GGUF'

if os.path.exists(CLEAN_DIR):
    print(f'Uploading merged HF checkpoint to {MERGED_REPO_ID}...')
    try:
        api.create_repo(repo_id=MERGED_REPO_ID, repo_type='model', exist_ok=True)
        api.upload_folder(
            folder_path=CLEAN_DIR,
            repo_id=MERGED_REPO_ID,
            repo_type='model',
            commit_message='Gemma 4 E4B legal merged safetensors checkpoint',
        )
        print(f'Uploaded merged checkpoint: https://huggingface.co/{MERGED_REPO_ID}')
    except Exception as e:
        print(f'Merged checkpoint upload failed: {e}')
else:
    print(f'Skipping merged upload: {CLEAN_DIR}/ not found')

if os.path.exists(GGUF_DIR):
    print(f'\nUploading optional GGUF artifacts to {GGUF_REPO_ID}...')
    try:
        api.create_repo(repo_id=GGUF_REPO_ID, repo_type='model', exist_ok=True)
        api.upload_folder(
            folder_path=GGUF_DIR,
            repo_id=GGUF_REPO_ID,
            repo_type='model',
            commit_message='Gemma 4 E4B legal VLM GGUF artifacts',
        )
        print(f'Uploaded GGUF: https://huggingface.co/{GGUF_REPO_ID}')
    except Exception as e:
        print(f'GGUF upload failed: {e}')
else:
    print(f'\nSkipping GGUF upload: {GGUF_DIR}/ not found')